# Structured output

Assistant version 3: Extract ticket fields into CSV and measure labelled-subset accuracy.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --group authoring`, then `uv run jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## Introduce Pydantic v2; with_structured_output; enums, nested models, optional fields

Predict the output before running. The function definitions below match `examples/01_pydantic.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Validate nested ticket data with Pydantic enums and nullable identifiers.

Run: uv run python modules/03_structured_output/examples/01_pydantic.py --offline
Expected: A TriageEnvelope containing shipping, low, positive and identifiers.
Cost: One short hosted call, typically under $0.01; --offline is free.
"""

from common.data import load_jsonl
from common.errors import run_cli
from common.fixtures import structured_fixture
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text
from common.schemas import TriageEnvelope
from common.triage import fixture_triage


def main() -> None:
    settings = lesson_settings()
    text = load_jsonl("support_tickets.jsonl")[0]["text"]
    fixture = structured_fixture(
        TriageEnvelope, {"triage": fixture_triage(text), "evidence": "Where is my shipment?"}
    )
    # KEY IDEA: Pydantic is a Python data validator; validation checks shape, not truth.
    structured = get_chat_model(settings=settings, responses=[fixture]).with_structured_output(
        TriageEnvelope
    )
    result = structured.invoke("Extract ticket fields and a short evidence quote: " + text)
    print_text(result.model_dump_json(indent=2))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## TypedDict and JSON Schema; with_structured_output

Predict the output before running. The function definitions below match `examples/02_schema_formats.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Compare TypedDict and JSON Schema contracts with explicit local validation.

Run: uv run python modules/03_structured_output/examples/02_schema_formats.py --offline
Expected: Two parsed dictionaries; TypeAdapter validates the TypedDict result.
Cost: Two short hosted calls; offline/local no API fee.
"""

from pydantic import TypeAdapter
from typing_extensions import TypedDict

from common.errors import run_cli
from common.fixtures import structured_fixture
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text


class ProductFact(TypedDict):
    """A product code and name copied from supplied facts."""

    sku: str
    name: str


SCHEMA = {
    "title": "ProductFactJSON",
    "description": "Extract a supplied product fact.",
    "type": "object",
    "properties": {"sku": {"type": "string"}, "name": {"type": "string"}},
    "required": ["sku", "name"],
    "additionalProperties": False,
}


def main() -> None:
    settings = lesson_settings()
    for schema in (ProductFact, SCHEMA):
        fixture = structured_fixture(schema, {"sku": "BW-001", "name": "Ridge Tent"})
        result = (
            get_chat_model(settings=settings, responses=[fixture])
            .with_structured_output(schema)
            .invoke("Extract: product code BW-001; product name Ridge Tent.")
        )
        # KEY IDEA: dict output is not a Pydantic instance; validate before business use.
        print_text(str(TypeAdapter(ProductFact).validate_python(result, strict=True)))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Provider-native JSON schema versus tool-calling method; capability checks

Predict the output before running. The function definitions below match `examples/03_output_methods.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Select provider-native JSON schema or a schema represented as a tool call.

Run: uv run python modules/03_structured_output/examples/03_output_methods.py --offline
Expected: Both methods parse the same supplied fixture; provider enforcement is not simulated.
Cost: Two hosted calls; model capability is required. See provider capabilities guide.
"""

from common.errors import run_cli
from common.fixtures import structured_fixture
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text
from common.schemas import TicketTriage
from common.triage import fixture_triage


def main() -> None:
    settings = lesson_settings()
    text = "Where is my shipment? Order BW1001, item BW-001. No rush."
    for method in ("json_schema", "function_calling"):
        fixture = structured_fixture(TicketTriage, fixture_triage(text))
        model = get_chat_model(settings=settings, responses=[fixture])
        # KEY IDEA: native constraints are provider-side; a tool schema describes arguments.
        result = model.with_structured_output(TicketTriage, method=method).invoke(text)
        print_text(f"{method}: {result.model_dump_json()}")
    if settings.course_offline:
        print_text(
            "Both fixtures used LangChain's tool parser; "
            "native provider enforcement was not tested."
        )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## ValidationError; invalid output, bounded repair and retry; deterministic fixtures

Predict the output before running. The function definitions below match `examples/04_validation_retry.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Repair invalid structured data with a bounded, schema-aware retry.

Run: uv run python modules/03_structured_output/examples/04_validation_retry.py --offline
Expected: An invalid priority is rejected, then repaired on the second fixture response.
Cost: At most two calls in this demo; offline fixtures intentionally fail first.
"""

from langchain_core.language_models import BaseChatModel
from langchain_core.messages import HumanMessage
from pydantic import ValidationError

from common.errors import run_cli
from common.fixtures import structured_fixture
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text
from common.schemas import TicketTriage
from common.triage import fixture_triage


def extract_with_repair(model: BaseChatModel, text: str, max_attempts: int = 2) -> TicketTriage:
    if not 1 <= max_attempts <= 3:
        raise ValueError("Use 1..3 attempts.")
    runnable = model.with_structured_output(TicketTriage)
    messages = [HumanMessage("Extract Brightwave ticket fields: " + text)]
    for attempt in range(max_attempts):
        try:
            result = runnable.invoke(messages)
            if result is None:
                raise ValueError("No structured result returned.")
            return result
        except ValidationError as error:
            if attempt + 1 == max_attempts:
                raise
            fields = sorted({str(item["loc"][0]) for item in error.errors()})
            # KEY IDEA: repair changes the prompt; a blind transport retry does not.
            messages.append(
                HumanMessage(
                    f"Your last extraction failed validation for {fields}. "
                    "Return the full schema; priority must be low, medium or high."
                )
            )
    raise RuntimeError("Repair attempts exhausted.")


def main() -> None:
    settings = lesson_settings()
    text = "Where is my shipment? No rush. Order BW1001, item BW-001."
    good = fixture_triage(text)
    bad = {**good, "priority": "yesterday"}
    model = get_chat_model(
        settings=settings,
        responses=[structured_fixture(TicketTriage, bad), structured_fixture(TicketTriage, good)],
    )
    print_text(extract_with_repair(model, text).model_dump_json())


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## StrOutputParser, PydanticOutputParser; when parsers remain useful

Predict the output before running. The function definitions below match `examples/05_parsers.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Parse text after generation when the model has no native schema interface.

Run: uv run python modules/03_structured_output/examples/05_parsers.py --offline
Expected: A plain string and a validated TicketTriage object.
Cost: One short hosted call, typically under $0.01; --offline is free.
"""

import json

from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text
from common.schemas import TicketTriage
from common.triage import fixture_triage


def main() -> None:
    settings = lesson_settings()
    text = "Where is my shipment? No rush."
    parser = PydanticOutputParser(pydantic_object=TicketTriage)
    model = get_chat_model(settings=settings, responses=[json.dumps(fixture_triage(text))])
    message = model.invoke(parser.get_format_instructions() + "\nTicket: " + text)
    # KEY IDEA: a parser validates after generation; it cannot force valid generation.
    print_text(StrOutputParser().invoke(message))
    print_text(parser.invoke(message).model_dump_json())


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Extract ticket fields into CSV and measure labelled-subset accuracy

Predict the output before running. The function definitions below match `use_case/ticket_triage.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Extract every ticket into CSV and score only the labelled subset.

Run: uv run python modules/03_structured_output/use_case/ticket_triage.py --offline
Expected: 100 rows in .runtime/triage.csv and separately labelled fixture/live metrics.
Cost: 100 short calls; budget up to $0.10 on the default low-cost profile. Run --offline first.
"""

import json

from common.config import ROOT
from common.data import load_jsonl
from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.triage import score_predictions, triage_ticket, write_triage_csv


def main() -> None:
    settings = lesson_settings()
    tickets = load_jsonl("support_tickets.jsonl")
    rows = []
    for ticket in tickets:
        # KEY IDEA: gold labels never enter the model prompt.
        prediction = triage_ticket(ticket["text"], settings)
        rows.append({"ticket_id": ticket["ticket_id"], **prediction.model_dump(mode="json")})
    output = ROOT / settings.runtime_dir / "triage.csv"
    write_triage_csv(rows, output)
    label = (
        "FIXTURE plumbing score (not model accuracy)"
        if settings.course_offline
        else "LIVE model accuracy"
    )
    print_text(f"{len(rows)} rows -> {output}")
    print_text(label + ": " + json.dumps(score_predictions(rows, tickets)))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run pytest modules/03_structured_output/tests` for the supplied solution checks.